# Baseline: four couplings under identical training

Reproduces the comparison of the four canonical couplings
(``independent``, ``hungarian_exact_ot``, ``sinkhorn_sampled``,
``sinkhorn_barycentric``) under the same MLP, optimizer, training
budget and rollout solver.

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np
import yaml

from otfm.couplings import (
    hungarian_pairing, independent_pairing,
    sample_pairs_from_coupling, sinkhorn_barycentric_pairing,
    sinkhorn_coupling,
)
from otfm.datasets import sample_8gaussians, sample_moons
from otfm.metrics import (
    empirical_w2_squared_hungarian, normalized_path_energy,
    path_energy, trajectory_curvature_metrics, velocity_variance_kernel,
)
from otfm.model import init_mlp_params
from otfm.plotting import plot_generated_vs_target, plot_loss_curves, plot_low_nfe_curves
from otfm.runtime import make_t_schedule, rollout_euler, rollout_euler_final, train_on_fixed_pairs


In [ ]:
cfg = yaml.safe_load(open('../configs/base.yaml'))
n = cfg['data']['n']
eps = cfg['sinkhorn']['default_eps']
train_steps = cfg['training']['train_steps']
rollout_steps = cfg['rollout']['default_steps']
widths = cfg['model']['widths']
nfe_list = cfg['rollout']['nfe_list']


In [ ]:
key = jax.random.PRNGKey(0)
key, k0, k1, kpair, kinit = jax.random.split(key, 5)

x0 = sample_moons(k0, n)
x1 = sample_8gaussians(k1, n)

P, _ = sinkhorn_coupling(x0, x1, epsilon=eps)
couplings = {
    'independent':           independent_pairing(x0, x1),
    'hungarian_exact_ot':    hungarian_pairing(x0, x1),
    f'sinkhorn_eps_{eps}':   sample_pairs_from_coupling(kpair, x0, x1, P),
    'sinkhorn_barycentric':  sinkhorn_barycentric_pairing(x0, x1, P),
}

base_params = init_mlp_params(kinit, widths)
t_schedule = make_t_schedule(seed=cfg['training']['t_schedule_seed'], steps=train_steps, n=n)


In [ ]:
results = {}
for name, (xa, xb) in couplings.items():
    params_tr, losses = train_on_fixed_pairs(base_params, xa, xb, t_schedule)
    traj = rollout_euler(params_tr, x0[:200], steps=rollout_steps)
    results[name] = {'params': params_tr, 'losses': losses, 'traj': traj}


In [ ]:
plot_loss_curves({name: r['losses'] for name, r in results.items()})
plt.show()


In [ ]:
plot_generated_vs_target(results, x1)
plt.show()


In [ ]:
import pandas as pd
x0_eval, x1_eval = x0, x1
w2_data = empirical_w2_squared_hungarian(x0_eval, x1_eval)

rows, low_nfe_rows = [], []
key_v = jax.random.PRNGKey(7)
for name, r in results.items():
    pe = path_energy(r['params'], x0_eval, steps=cfg['rollout']['pe_steps'])
    npe = normalized_path_energy(pe, w2_data)
    for nfe in nfe_list:
        x_gen = rollout_euler_final(r['params'], x0_eval, steps=nfe)
        low_nfe_rows.append({
            'coupling': name, 'nfe': nfe,
            'endpoint_w2_sq': empirical_w2_squared_hungarian(x_gen, x1_eval),
        })
    key_v, kv = jax.random.split(key_v)
    xa, xb = couplings[name]
    target_var = velocity_variance_kernel(
        xa, xb, **{k: cfg['metrics']['velocity_kernel'][k]
                   for k in ('n_time_samples', 'sigma', 'n_query', 'n_ref', 'include_time', 'time_scale')},
        key=kv,
    )
    curv = trajectory_curvature_metrics(r['params'], x0_eval, steps=cfg['metrics']['curvature_steps'])
    rows.append({
        'coupling': name, 'PE': pe, 'NPE': npe,
        'loss_var': float(np.var(r['losses'])),
        'target_velocity_kernel_var': target_var,
        **curv,
    })

summary_df = pd.DataFrame(rows).sort_values('coupling').reset_index(drop=True)
low_nfe_df = pd.DataFrame(low_nfe_rows)
summary_df


In [ ]:
plot_low_nfe_curves(low_nfe_df)
plt.show()
